# Lab 3.2 &mdash; A Multi-Step Workflow

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 1 &middot; Module 3 &mdash; LangGraph: Stateful Agent Workflows**

### What you'll do
- Chain three nodes so each reads what the last one wrote
- Discover that the <i>order</i> of your edges is a real design decision
- Watch a run with <code>stream()</code>, one node at a time

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All eight Module 3 labs work one case: leave requests in a small HR
> system. The rules are ordinary on purpose &mdash; the only new thing here is LangGraph.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-3-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# Leave requests in a small HR system. Ordinary rules on purpose: the only new thing in these
# eight labs is LangGraph. One flat dict -- no joins, no helpers, nothing to learn here.

REQUESTS = {
    "LV-5001": {"who": "Priya Nair",   "days":  3, "kind": "annual", "reason": "family wedding",
                "balance": 12, "manager": "Devi R."},
    "LV-5002": {"who": "Rahul Menon",  "days":  5, "kind": "annual", "reason": "",
                "balance":  3, "manager": "Devi R."},
    "LV-5003": {"who": "Anita Sharma", "days":  2, "kind": "annual", "reason": "moving house",
                "balance":  0, "manager": "Sam O."},
    "LV-5004": {"who": "Vikram Rao",   "days": 15, "kind": "annual", "reason": "sabbatical",
                "balance": 20, "manager": "Sam O."},
    "LV-5005": {"who": "Priya Nair",   "days":  1, "kind": "sick",   "reason": "flu",
                "balance": 12, "manager": "Devi R."},
}

# The handbook, as three numbers. Every routing decision in this module comes from these.
POLICY = {"manager_over_days": 2, "hr_over_days": 10, "max_clarifications": 2}

print(len(REQUESTS), "leave requests loaded")

## Concept

A node can read anything an earlier node wrote &mdash; the state is the channel between them.

That gives you something a message list never did: a **dependency order**. `draft` cannot run
before `check`, because it reads a field `check` creates. In a conversation that constraint lives
in your head. In a graph it lives in the edges, and breaking it is a `KeyError` on the first run
rather than a confident answer built on a field nobody set.

## Section 1 &mdash; Three nodes

`check` never looks at `REQUESTS`. It reads `days` and `balance` out of the **state**, because
`summarise` put them there.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class LeaveState(TypedDict):
    request_id: str
    days: int
    balance: int
    covered: bool        # created by check
    needs_manager: bool  # created by check
    decision: str        # created by draft
    notes: list


def summarise(state: LeaveState) -> dict:
    r = REQUESTS[state["request_id"]]
    return {"days": r["days"], "balance": r["balance"],
            "notes": state["notes"] + ["summarise"]}


def check_request(state: LeaveState) -> dict:
    return {"covered": state["balance"] >= state["days"],
            "needs_manager": state["days"] > POLICY["manager_over_days"],
            "notes": state["notes"] + ["check"]}


def draft(state: LeaveState) -> dict:
    if not state["covered"]:
        text = "declined: not enough balance"
    elif state["needs_manager"]:
        text = f'pending: {state["days"]} days needs manager approval'
    else:
        text = "approved automatically"
    return {"decision": text, "notes": state["notes"] + ["draft"]}

In [ ]:
# --- Self-check: Section 1   (the middle node on its own)
check("2 days does not need a manager, 3 days does",
      lambda: check_request({"days": 2, "balance": 9, "notes": []})["needs_manager"] is False
          and check_request({"days": 3, "balance": 9, "notes": []})["needs_manager"] is True,
      "POLICY['manager_over_days'] is 2, and the comparison is strictly greater than")
check("a 5-day request against a 3-day balance is not covered",
      lambda: check_request({"days": 5, "balance": 3, "notes": []})["covered"] is False)
score()

## Section 2 &mdash; The edges that impose the order

`draft` reads `covered` and `needs_manager` &mdash; fields that **do not exist** until `check` has
run. The edges are what guarantee they are there.

In [ ]:
def build_chain():
    builder = StateGraph(LeaveState)
    builder.add_node("summarise", summarise)
    builder.add_node("check", check_request)
    builder.add_node("draft", draft)

    builder.add_edge(START, "summarise")
    builder.add_edge("summarise", "check")   # covered / needs_manager are created here
    builder.add_edge("check", "draft")       # ...and read here
    builder.add_edge("draft", END)
    return builder.compile()

In [ ]:
# --- Self-check: Section 2   (three real nodes in a real compiled graph)
def run(rid):
    return build_chain().invoke({"request_id": rid, "notes": []})

check("all three nodes ran, in dependency order",
      lambda: run("LV-5001")["notes"] == ["summarise", "check", "draft"],
      "notes is the run order -- a missing node means a missing edge")
check("LV-5003 declined, LV-5005 auto-approved, LV-5004 pending",
      lambda: (run("LV-5003")["decision"].startswith("declined")
               and run("LV-5005")["decision"] == "approved automatically"
               and run("LV-5004")["decision"].startswith("pending")))
score()

## Watch it run

`stream()` gives you the run a node at a time instead of only the final state &mdash; the view you
want when a workflow does something you did not expect.

In [ ]:
app = guard(build_chain)

if app is not None:
    for rid in ["LV-5001", "LV-5003", "LV-5004"]:
        print(f"\n=== {rid} ===")
        for step in app.stream({"request_id": rid, "notes": []}):
            for node, update in step.items():
                print(f"  {node:10} wrote: {', '.join(k for k in update if k != 'notes')}")

### Read it

`summarise` produced `days` and `balance`; `check` consumed them and produced `covered` and
`needs_manager`; `draft` consumed those. The state is a pipeline and every arrow in it is one
`add_edge` line &mdash; which is why swapping two edges gives you `KeyError: 'covered'` on the
first run rather than a plausible answer later.

In [ ]:
score()

## Your turn

1. Wire `summarise -> draft -> check` deliberately and run it. That crash is the design working.
2. `LV-5002` has an empty `reason`. Add a `validate` node at the front writing `complete: bool`,
   and have `draft` refuse when it is false. You have just invented Lab 3.5's loop.